In [ ]:
#Weekly seasonality & holidays added.
!pip install prophet cmdstanpy --upgrade

In [ ]:
import pandas as pd
from prophet import Prophet
from google.colab import files

# === Upload file ===
uploaded = files.upload()
file_path = list(uploaded.keys())[0]
print("Uploaded file:", file_path)

# === Load & prepare data ===
df = pd.read_excel(file_path)
df["Date"] = pd.to_datetime(df["Date"])

# Melt to long format
df_long = pd.melt(df, id_vars=["Date"], var_name="Store", value_name="Sales")

# Forecast parameters
forecast_start = "2026-05-04"
forecast_end = "2026-05-10"
future_dates = pd.date_range(start=forecast_start, end=forecast_end, freq='D')
store_forecasts = {}

# === Loop over stores and forecast ===
for store in df_long["Store"].unique():
    store_df = df_long[df_long["Store"] == store].dropna()

    # Prophet requires columns named 'ds' and 'y'
    prophet_df = store_df.rename(columns={"Date": "ds", "Sales": "y"})

    model = Prophet(weekly_seasonality=True, yearly_seasonality=True)
    model.add_country_holidays(country_name='US')
    model.fit(prophet_df)

    # Create future dataframe
    future = pd.DataFrame({"ds": future_dates})

    # Forecast
    forecast = model.predict(future)
    store_forecasts[store] = forecast[["ds", "yhat"]].rename(columns={"yhat": store})

# === Combine forecasts into wide format ===
wide_forecast = pd.DataFrame({"Date": future_dates})
for store, store_df in store_forecasts.items():
    wide_forecast = wide_forecast.merge(store_df.rename(columns={"ds": "Date"}), on="Date", how="left")

# === Save and download ===
output_file = "Prophet_Forecast_Wide.xlsx"
wide_forecast.to_excel(output_file, index=False)
files.download(output_file)

print("✅ Forecast complete. Downloading:", output_file)


INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


Saving Wings Actual..xlsx to Wings Actual..xlsx
Uploaded file: Wings Actual..xlsx


INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.
INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Forecast complete. Downloading: Prophet_Forecast_Wide.xlsx
